In [0]:
%sql
--quais países estão conseguindo crescer economicamente enquanto reduzem sua pegada ambiental?

CREATE OR REPLACE VIEW portfolio_energia.gold.vw_pib_vs_emissoes AS
WITH emissao_anual_ember AS (
    SELECT 
        fc.id_location,
        dd.year,
        SUM(fc.emission) AS total_emissao_ano
    FROM portfolio_energia.gold.fact_carbon_emission AS fc
    INNER JOIN portfolio_energia.gold.dim_date AS dd 
        ON fc.date_id = dd.date_id
    GROUP BY fc.id_location, dd.year
)

SELECT 
    dl.country AS pais,
    dd.year AS ano,
    fe.metric AS pib_per_capita,
    ea.total_emissao_ano AS intensidade_emissao
FROM portfolio_energia.gold.fact_economic AS fe
INNER JOIN portfolio_energia.gold.dim_location AS dl 
    ON fe.location_id = dl.id_location
INNER JOIN portfolio_energia.gold.dim_date AS dd 
    ON fe.date_id = dd.date_id
INNER JOIN portfolio_energia.gold.dim_indicator AS di 
    ON fe.indicator_id = di.indicator_id
INNER JOIN emissao_anual_ember AS ea 
    ON fe.location_id = ea.id_location 
    AND dd.year = ea.year
WHERE 
    di.indicator_code = 'NY.GDP.PCAP.CD' AND dd.year BETWEEN 2010 AND 2024;

--a segunda análise cruze o indicador percentual de acesso à eletricidade com o volume absoluto da demanda de eletricidade

CREATE OR REPLACE VIEW portfolio_energia.gold.vw_acesso_vs_demanda AS
WITH demanda_anual_ember AS (
    SELECT 
        felec.id_location,
        dd.year,
        SUM(felec.demand) AS demanda_total_ano_twh
    FROM portfolio_energia.gold.fact_electricity AS felec
    INNER JOIN portfolio_energia.gold.dim_date AS dd 
        ON felec.date_id = dd.date_id
    GROUP BY felec.id_location, dd.year
)

SELECT 
    dl.country AS pais,
    dd.year AS ano,
    fe.metric AS percentual_acesso,
    da.demanda_total_ano_twh
FROM portfolio_energia.gold.fact_economic AS fe
INNER JOIN portfolio_energia.gold.dim_location AS dl 
    ON fe.location_id = dl.id_location
INNER JOIN portfolio_energia.gold.dim_date AS dd 
    ON fe.date_id = dd.date_id
INNER JOIN portfolio_energia.gold.dim_indicator AS di 
    ON fe.indicator_id = di.indicator_id
INNER JOIN demanda_anual_ember AS da 
    ON fe.location_id = da.id_location 
    AND dd.year = da.year
WHERE 
    di.indicator_code = 'EG.ELC.ACCS.ZS' AND dd.year BETWEEN 2010 AND 2024;


-- investimento estrangeiro direto (FDI) vs. intensidade de emissões


CREATE OR REPLACE VIEW portfolio_energia.gold.vw_fdi_vs_emissoes AS
WITH emissao_anual_ember AS (
    SELECT 
        fc.id_location,
        dd.year,
        ROUND(AVG(fc.emission))  AS intensidade_emissao_media
    FROM portfolio_energia.gold.fact_carbon_emission AS fc
    INNER JOIN portfolio_energia.gold.dim_date AS dd 
        ON fc.date_id = dd.date_id
    GROUP BY fc.id_location, dd.year
)

SELECT 
    dl.country AS pais,
    dd.year AS ano,
    fe.metric AS fdi_entrada_liquida,
    ea.intensidade_emissao_media
FROM portfolio_energia.gold.fact_economic AS fe
INNER JOIN portfolio_energia.gold.dim_location AS dl 
    ON fe.location_id = dl.id_location
INNER JOIN portfolio_energia.gold.dim_date AS dd 
    ON fe.date_id = dd.date_id
INNER JOIN portfolio_energia.gold.dim_indicator AS di 
    ON fe.indicator_id = di.indicator_id
INNER JOIN emissao_anual_ember AS ea 
    ON fe.location_id = ea.id_location 
    AND dd.year = ea.year
WHERE 
    di.indicator_code = 'BX.KLT.DINV.WD.GD.ZS' AND dd.year BETWEEN 2010 AND 2024; 

-- Geografia dos Data Centers vs. Matriz Energética

CREATE OR REPLACE VIEW portfolio_energia.gold.vw_data_center_vs_emissoes AS
WITH emissao_anual_ember AS (
    SELECT 
        fc.id_location,
        dd.year,
        ROUND(AVG(fc.emission)) AS intensidade_emissao_media
    FROM portfolio_energia.gold.fact_carbon_emission AS fc
    INNER JOIN portfolio_energia.gold.dim_date AS dd 
        ON fc.date_id = dd.date_id
    GROUP BY fc.id_location, dd.year
),
contagem_data_centers AS (
    SELECT 
        iso2,
        COUNT(id) AS total_data_centers
    FROM portfolio_energia.gold.dim_data_center
    GROUP BY iso2
)

SELECT 
    dl.country AS pais,
    ea.year AS ano,
    COALESCE(cdc.total_data_centers, 0) AS qtd_data_centers,
    ea.intensidade_emissao_media
FROM emissao_anual_ember AS ea
INNER JOIN portfolio_energia.gold.dim_location AS dl 
    ON ea.id_location = dl.id_location
LEFT JOIN contagem_data_centers AS cdc
    ON dl.iso2 = cdc.iso2
WHERE ea.year BETWEEN 2010 AND 2024;



In [0]:
%uv pip install seaborn

import matplotlib.pyplot as plt
import seaborn as sns

df = spark.table("portfolio_energia.gold.vw_pib_vs_emissoes").toPandas()

paises_foco = [
    "Brazil",
    "United States",
    "Germany",
    "France",
    "Japan",
    "Canada",
    "Italy",
    "India",
]
df_foco = df[df["pais"].isin(paises_foco)]

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(
    data=df_foco,
    x="ano",
    y="pib_per_capita",
    hue="pais",
    linewidth=2.5,
    ax=axes[0],
)
axes[0].set_title(
    "Evolução do PIB per Capita (USD)", fontsize=13, fontweight="bold"
)
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("PIB per Capita (US$)")

sns.lineplot(
    data=df_foco,
    x="ano",
    y="intensidade_emissao",
    hue="pais",
    linewidth=2.5,
    ax=axes[1],
)
axes[1].set_title(
    "Evolução das Emissões de Carbono", fontsize=13, fontweight="bold"
)
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Total de Emissões")

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

df_acesso = spark.table("portfolio_energia.gold.vw_acesso_vs_demanda").toPandas()

paises_foco = [
    "Brazil",
    "United States of America",
    "China",
    "India",
    "South Africa",
    "Nigeria",
    "Germany",
]
df_acesso_foco = df_acesso[df_acesso["pais"].isin(paises_foco)]

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(
    data=df_acesso_foco,
    x="ano",
    y="percentual_acesso",
    hue="pais",
    linewidth=2.5,
    ax=axes[0],
)
axes[0].set_title(
    "Acesso à Eletricidade (% da População)", fontsize=13, fontweight="bold"
)
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("Acesso à Eletricidade (%)")
axes[0].set_ylim(0, 105)

sns.lineplot(
    data=df_acesso_foco,
    x="ano",
    y="demanda_total_ano_twh",
    hue="pais",
    linewidth=2.5,
    ax=axes[1],
)
axes[1].set_title(
    "Evolução da Demanda Total de Eletricidade (TWh)", fontsize=13, fontweight="bold"
)
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Demanda Total (TWh)")

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

df_fdi = spark.table("portfolio_energia.gold.vw_fdi_vs_emissoes").toPandas()

paises_foco = [
    "Brazil",
    "United States of America",
    "Germany",
    "France",
    "Japan",
    "India",
]
df_fdi_foco = df_fdi[df_fdi["pais"].isin(paises_foco)]

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(
    data=df_fdi_foco,
    x="ano",
    y="fdi_entrada_liquida",
    hue="pais",
    linewidth=2.5,
    ax=axes[0],
)
axes[0].set_title(
    "Evolução do Investimento Estrangeiro Direto (% PIB)", fontsize=13, fontweight="bold"
)
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("FDI Entrada Líquida (% do PIB)")

sns.scatterplot(
    data=df_fdi_foco,
    x="intensidade_emissao_media",
    y="fdi_entrada_liquida",
    hue="pais",
    s=90,
    alpha=0.8,
    ax=axes[1],
)
axes[1].set_title(
    "FDI vs. Intensidade Média de Emissões", fontsize=13, fontweight="bold"
)
axes[1].set_xlabel("Intensidade Média de Emissões")
axes[1].set_ylabel("FDI Entrada Líquida (% do PIB)")

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

df_dc = spark.table("portfolio_energia.gold.vw_data_center_vs_emissoes").toPandas()

ano_mais_recente = df_dc["ano"].max()
df_dc_recente = (
    df_dc[df_dc["ano"] == ano_mais_recente]
    .sort_values(by="qtd_data_centers", ascending=False)
    .head(10)
)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(
    data=df_dc_recente,
    x="qtd_data_centers",
    y="pais",
    hue="pais",
    palette="Blues_r",
    legend=False,
    ax=axes[0],
)
axes[0].set_title(
    f"Top 10 Países em Qtd. de Data Centers ({ano_mais_recente})",
    fontsize=13,
    fontweight="bold",
)
axes[0].set_xlabel("Quantidade de Data Centers")
axes[0].set_ylabel("País")

sns.barplot(
    data=df_dc_recente,
    x="intensidade_emissao_media",
    y="pais",
    hue="pais",
    palette="Reds_r",
    legend=False,
    ax=axes[1],
)
axes[1].set_title(
    f"Intensidade de Emissões da Matriz ({ano_mais_recente})",
    fontsize=13,
    fontweight="bold",
)
axes[1].set_xlabel("Intensidade Média de Emissões")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()